In [1]:
!git clone https://github.com/mohamadghoroobi/tkh-hierarchy-project

Cloning into 'tkh-hierarchy-project'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 20 (delta 4), reused 15 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 263.08 KiB | 2.02 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [5]:
%cd /content/tkh-hierarchy-project

/content/tkh-hierarchy-project


In [6]:
!ls

data  LICENSE  notebooks  README.md


# Pre-Operations

## Imports

In [7]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

## Define paths

In [8]:
PROJECT_DIR = Path("/content/tkh-hierarchy-project")

DATA_DIR = PROJECT_DIR / "data"

TKH_PATH = DATA_DIR / "tkh_collection10.json"
QUESTIONS_PATH = DATA_DIR / "questions.csv"
GOLD_PATH = DATA_DIR / "ground_truth.json"
ARTICLES_PATH = DATA_DIR / "collection10_articles.csv"


print(TKH_PATH)
print(QUESTIONS_PATH)
print(GOLD_PATH)
print(ARTICLES_PATH)

/content/tkh-hierarchy-project/data/tkh_collection10.json
/content/tkh-hierarchy-project/data/questions.csv
/content/tkh-hierarchy-project/data/ground_truth.json
/content/tkh-hierarchy-project/data/collection10_articles.csv


### Load TKH data

In [9]:
with open(
    TKH_PATH,
    "r",
    encoding="utf-8"
) as f:

    tkh = json.load(f)


nodes = tkh["nodes"]
hyperedges = tkh["hyperedges"]


print(
    f"Nodes: {len(nodes):,}"
)

print(
    f"Hyperedges: {len(hyperedges):,}"
)

Nodes: 5,798
Hyperedges: 1,429


# SECTION 2 — Temporally Honest Snapshot Construction

This section constructs yearly TKH snapshots while avoiding temporal leakage.

A node is visible at time t only if first_seen_year <= t.
A hyperedge is visible only if all participating nodes are available.

## Define snapshot years

In [10]:
SNAPSHOT_YEARS = [
    2020,
    2022,
    2024,
    2026,
]

SNAPSHOT_YEARS

[2020, 2022, 2024, 2026]

## Create node visibility function

In [11]:
def node_available_at_time(node, year):
    """
    Determines whether a node exists in the TKH snapshot.

    Temporal rule:
    first_seen_year <= snapshot year
    """

    return (
        node.get("first_seen_year") is not None
        and node["first_seen_year"] <= year
    )

### Test

In [12]:
test_node = nodes[0]

print(test_node["surface_form"])
print(test_node["first_seen_year"])

for y in SNAPSHOT_YEARS:
    print(
        y,
        node_available_at_time(test_node, y)
    )

Neural Message Passing for Quantum Chemistry
2017
2020 True
2022 True
2024 True
2026 True


## Build node snapshots

In [13]:
def build_node_snapshot(nodes, year):

    snapshot_nodes = [
        node
        for node in nodes
        if node_available_at_time(node, year)
    ]

    return snapshot_nodes

In [14]:
node_snapshots = {}

for year in SNAPSHOT_YEARS:

    snapshot_nodes = build_node_snapshot(
        nodes,
        year
    )

    node_snapshots[year] = snapshot_nodes

    print(
        year,
        len(snapshot_nodes)
    )

2020 1505
2022 2164
2024 4164
2026 5798


## Create node lookup per snapshot

In [15]:
node_id_snapshots = {}

for year, snapshot_nodes in node_snapshots.items():

    node_id_snapshots[year] = {
        node["id"]
        for node in snapshot_nodes
    }


for year in SNAPSHOT_YEARS:
    print(
        year,
        len(node_id_snapshots[year])
    )

2020 1505
2022 2164
2024 4164
2026 5798


## Hyperedge availability rule

We do not truncate hyperedges.

The previous notebook did:

edge:
A,B,C,D

snapshot:
A,B

→ creates A,B

That changes the meaning.

Instead:

A hyperedge exists only if all members exist.

In [17]:
def edge_available_at_time(edge, available_node_ids):

    members = edge["members"]

    return all(
        member in available_node_ids
        for member in members
    )

## Build hyperedge snapshots

In [18]:
def build_edge_snapshot(
    hyperedges,
    available_node_ids
):

    snapshot_edges = [
        edge
        for edge in hyperedges
        if edge_available_at_time(
            edge,
            available_node_ids
        )
    ]

    return snapshot_edges

In [19]:
edge_snapshots = {}

for year in SNAPSHOT_YEARS:

    snapshot_edges = build_edge_snapshot(
        hyperedges,
        node_id_snapshots[year]
    )

    edge_snapshots[year] = snapshot_edges

    print(
        year,
        len(snapshot_edges)
    )

2020 374
2022 529
2024 983
2026 1429


## Snapshot summary table

Now create our first temporal report.

In [20]:
snapshot_summary = []

for year in SNAPSHOT_YEARS:

    snapshot_summary.append({

        "year": year,

        "nodes":
            len(node_snapshots[year]),

        "hyperedges":
            len(edge_snapshots[year]),

        "avg_arity":
            np.mean(
                [
                    len(e["members"])
                    for e in edge_snapshots[year]
                ]
            ),

        "multiway_fraction":
            np.mean(
                [
                    len(e["members"]) > 2
                    for e in edge_snapshots[year]
                ]
            )
    })


snapshot_summary_df = pd.DataFrame(
    snapshot_summary
)

snapshot_summary_df

,year,nodes,hyperedges,avg_arity,multiway_fraction
0,2020,1505,374,5.572193,0.708556
1,2022,2164,529,5.810964,0.735350
2,2024,4164,983,6.189217,0.786368
3,2026,5798,1429,6.248425,0.804759


## Check temporal monotonicity

In [21]:
for i in range(len(SNAPSHOT_YEARS)-1):

    y1 = SNAPSHOT_YEARS[i]
    y2 = SNAPSHOT_YEARS[i+1]

    assert (
        len(node_snapshots[y2])
        >=
        len(node_snapshots[y1])
    )

print(
    "Node growth monotonicity verified."
)

Node growth monotonicity verified.


In [22]:
for i in range(len(SNAPSHOT_YEARS)-1):

    y1 = SNAPSHOT_YEARS[i]
    y2 = SNAPSHOT_YEARS[i+1]

    assert (
        len(edge_snapshots[y2])
        >=
        len(edge_snapshots[y1])
    )

print(
    "Hyperedge growth monotonicity verified."
)

Hyperedge growth monotonicity verified.


## Compare against event-time interpretation

This is important for the report.

We will calculate the difference.

In [23]:
def event_time_nodes(nodes, year):

    return [
        n
        for n in nodes
        if n["year"] <= year
    ]


comparison = []

for year in SNAPSHOT_YEARS:

    comparison.append({

        "year": year,

        "availability_nodes":
            len(node_snapshots[year]),

        "event_time_nodes":
            len(event_time_nodes(nodes, year)),

        "difference":
            (
                len(event_time_nodes(nodes, year))
                -
                len(node_snapshots[year])
            )
    })


pd.DataFrame(comparison)

,year,availability_nodes,event_time_nodes,difference
0,2020,1505,1839,334
1,2022,2164,2496,332
2,2024,4164,4307,143
3,2026,5798,5798,0


This will likely show:

event_time_nodes > availability_nodes

especially in earlier years.

That is exactly the temporal leakage problem we identified.

In [24]:
import json

snapshot_metadata = {

    str(year): {

        "nodes":
            len(node_snapshots[year]),

        "hyperedges":
            len(edge_snapshots[year]),

        "temporal_definition":
            "first_seen_year <= snapshot_year"

    }

    for year in SNAPSHOT_YEARS
}


with open(
    "snapshot_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        snapshot_metadata,
        f,
        indent=2
    )


print(
    "Saved snapshot_metadata.json"
)

Saved snapshot_metadata.json


## Git Push

In [25]:
from pathlib import Path

CLEAN = Path("/content/drive/MyDrive/Apply/Germany/ConstructorLabs/02_temporal_snapshots.ipynb")
REPO_FILE = Path(
    "/content/tkh-hierarchy-project/"
    "notebooks/02_temporal_snapshots.ipynb"
)

print("Clean file exists:", CLEAN.exists())
print("Repo file exists :", REPO_FILE.exists())

Clean file exists: True
Repo file exists : False
